# SionnaRTStudio — ZIP-only Google Colab Launcher

Ovaj notebook pokreće SionnaRTStudio **isključivo iz ZIP arhive koju ti uploaduješ**. Ne klonira GitHub repo, ne radi `git pull` i ne zahtijeva GitHub pristup.

## Prije pokretanja

1. U Colab-u izaberi **Runtime → Change runtime type → T4 GPU** (ili drugi dostupni NVIDIA GPU).
2. Preporučujem **Runtime → Disconnect and delete runtime** prije prve upotrebe ovog notebooka, kako ne bi ostali stari serveri ili stari source fajlovi.
3. Pokreći ćelije redom od vrha prema dnu. U ćeliji **Upload project ZIP** izaberi jedan kompletan SionnaRTStudio source ZIP.
4. Nemoj ovdje uploadovati `exported.zip` sa učionicom. Taj ZIP se kasnije učitava kroz Indoor Scene importer u samoj aplikaciji.
5. Colab runtime je privremen; nakon isteka sesije notebook treba ponovo pokrenuti.

Notebook raspakuje ZIP u `/content/SionnaRTStudio`, pokreće `setup.sh` iz te arhive, kreira/koristi projektni `.venv`, instalira frontend zavisnosti iz `package.json`/`package-lock.json`, automatski pronalazi i izvršava sve `check:*` skripte definisane u `package.json`, zatim pokreće backend testove, TypeScript check i production build, pa tek onda pokreće `run.sh`. Nove faze mogu dodati nove `check:*` skripte bez pravljenja novog Colab notebooka. Lokalni `.venv` i `.node` mogu se sačuvati pri ponovnom uploadu druge ZIP verzije; `node_modules` se namjerno briše i instalira ponovo da ne bi ostala zastarjela frontend instalacija.

**Važno:** GitHub se nigdje ne koristi u ovom workflow-u. ZIP koji izabereš je jedini izvor izvornog koda za tu sesiju.

In [ ]:
# 1. Provjera runtime-a i sistemske zavisnosti
import os
import sys
import subprocess
import shutil
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Platforma:", sys.platform)

has_gpu = False
if shutil.which("nvidia-smi"):
    gpu_check = subprocess.run(["nvidia-smi", "-L"], text=True, capture_output=True)
    has_gpu = gpu_check.returncode == 0 and "GPU" in gpu_check.stdout
    print("nvidia-smi:")
    print((gpu_check.stdout or gpu_check.stderr).strip()[:2000])
else:
    print("nvidia-smi nije dostupan u ovom runtime-u.")

if has_gpu:
    print("\n✅ NVIDIA GPU je vidljiv. SionnaRTStudio će pokušati CUDA varijantu.")
else:
    print("\n⚠️ NVIDIA GPU nije potvrđen. Aplikacija će pokušati CPU/LLVM ray tracing; može biti znatno sporija.")

# SionnaRTStudio setup iz ZIP-a očekuje Node/npm, venv i LLVM fallback na sistemima bez CUDA.
# Ove pakete instaliramo na nivou Colab VM-a; Python paketi aplikacije idu u projektni .venv.
if not Path("/content/.srts_colab_system_deps_ok").exists():
    print("\nInstaliram sistemske pakete potrebne za setup.sh...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run([
        "apt-get", "install", "-y", "-qq",
        "curl", "ca-certificates", "tar", "xz-utils",
        "python3-venv", "llvm-runtime"
    ], check=True)
    Path("/content/.srts_colab_system_deps_ok").touch()
    print("✅ Sistemske zavisnosti su instalirane.")
else:
    print("✅ Sistemske zavisnosti su već instalirane u ovoj sesiji.")


In [ ]:
# 2. Upload project ZIP — bez GitHub clone/pull koraka
from google.colab import files
from pathlib import Path, PurePosixPath
import shutil
import zipfile
import stat

PROJECT_DIR = Path("/content/SionnaRTStudio")
TEMP_DIR = Path("/content/srts_zip_source_unpack")
PRESERVE = {".venv", ".node", "data"}
MAX_MEMBERS = 20000
MAX_UNCOMPRESSED_BYTES = 2 * 1024 * 1024 * 1024  # 2 GiB safety limit

print("Izaberi TAČNO JEDAN kompletan SionnaRTStudio source ZIP.")
print("Ne biraj exported.zip sa indoor scenom — taj fajl se kasnije uvozi kroz GUI.")
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if len(zip_names) != 1:
    raise ValueError(f"Očekivan je tačno jedan .zip fajl; pronađeno: {len(zip_names)}")

zip_name = zip_names[0]
zip_path = Path("/content") / zip_name
if not zip_path.exists():
    zip_path.write_bytes(uploaded[zip_name])

shutil.rmtree(TEMP_DIR, ignore_errors=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as archive:
    infos = archive.infolist()
    if len(infos) > MAX_MEMBERS:
        raise ValueError(f"ZIP ima previše stavki ({len(infos)}); limit je {MAX_MEMBERS}.")
    total_uncompressed = sum(info.file_size for info in infos)
    if total_uncompressed > MAX_UNCOMPRESSED_BYTES:
        raise ValueError(
            f"Raspakovani ZIP je prevelik ({total_uncompressed} bajtova); "
            f"limit je {MAX_UNCOMPRESSED_BYTES}."
        )

    # Reject path traversal, absolute paths, Windows drive paths and symlinks.
    for info in infos:
        normalized = info.filename.replace("\\", "/")
        member = PurePosixPath(normalized)
        if member.is_absolute() or ".." in member.parts or (member.parts and ":" in member.parts[0]):
            raise ValueError(f"ZIP sadrži nedozvoljenu putanju: {info.filename}")
        mode = (info.external_attr >> 16) & 0o170000
        if mode == stat.S_IFLNK:
            raise ValueError(f"ZIP sadrži simboličku vezu, odbijam ekstrakciju: {info.filename}")

    archive.extractall(TEMP_DIR)

# Support both ZIP layouts: source files at archive root, or one enclosing folder.
search_roots = [TEMP_DIR] + [p for p in TEMP_DIR.iterdir() if p.is_dir()]
project_candidates = [
    p for p in search_roots
    if (p / "setup.sh").is_file()
    and (p / "run.sh").is_file()
    and (p / "package.json").is_file()
    and (p / "backend" / "main.py").is_file()
]
if len(project_candidates) != 1:
    raise RuntimeError(
        "ZIP nije prepoznat kao kompletan SionnaRTStudio source arhiv. "
        f"Pronađeno je {len(project_candidates)} mogućih korijena projekta. "
        "Očekujem setup.sh, run.sh, package.json i backend/main.py u jednom folderu."
    )
SOURCE_DIR = project_candidates[0]
print("Prepoznat source folder:", SOURCE_DIR)

# Stop here if a previous project directory contains a live app: the user should
# start from a clean Colab runtime rather than overwrite files under a running server.
if PROJECT_DIR.exists():
    for child in list(PROJECT_DIR.iterdir()):
        if child.name in PRESERVE:
            continue
        if child.is_dir() and not child.is_symlink():
            shutil.rmtree(child)
        else:
            child.unlink()
else:
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# Copy the ZIP source. Preserve only runtime assets; deliberately DO NOT preserve
# node_modules, dist, .git, or any prior checked-out source files.
for item in SOURCE_DIR.iterdir():
    if item.name in {".venv", ".node", "node_modules", "dist", ".git"}:
        continue
    destination = PROJECT_DIR / item.name
    if item.name == "data" and destination.exists():
        print("Čuvam postojeći /content/SionnaRTStudio/data.")
        continue
    if item.is_dir():
        shutil.copytree(item, destination, dirs_exist_ok=True)
    else:
        shutil.copy2(item, destination)

required = ["setup.sh", "run.sh", "package.json", "backend/main.py", "src/main.tsx"]
missing = [name for name in required if not (PROJECT_DIR / name).is_file()]
if missing:
    raise FileNotFoundError("ZIP nije kopiran kompletno; nedostaju: " + ", ".join(missing))

# Capture an immutable fingerprint of the selected source package for diagnostics.
import hashlib
sha = hashlib.sha256(zip_path.read_bytes()).hexdigest()
print("\n✅ Source ZIP je učitan direktno — bez GitHub-a.")
print("ZIP:", zip_name)
print("SHA-256:", sha)
print("Project:", PROJECT_DIR)
print("Postojeći runtime direktorijumi:", [name for name in PRESERVE if (PROJECT_DIR / name).exists()])
print("node_modules postoji prije setup-a:", (PROJECT_DIR / "node_modules").exists())
if (PROJECT_DIR / "node_modules").exists():
    raise RuntimeError("Stari node_modules nije uklonjen; restartuj runtime i ponovo učitaj ZIP.")


In [ ]:
# 3. Instalacija originalnih SionnaRTStudio zavisnosti — sa dijagnostikom grešaka
# Koristi originalni setup.sh, ali sačuvaj puni izlaz u log i prikaži ga pri neuspjehu.
import os
import sys
import shutil
import subprocess
from pathlib import Path

PROJECT_DIR = Path("/content/SionnaRTStudio")
SETUP_LOG = Path("/content/SionnaRTStudio_setup.log")
if not (PROJECT_DIR / "setup.sh").exists():
    raise FileNotFoundError("Nije pronađen setup.sh. Prvo pokreni ćeliju 2 i uploaduj kompletan source ZIP.")

print("=== Predprovjera setup.sh ===")
print("Python aktivnog Colab kernela:", sys.version.split()[0], sys.executable)

# COLAB FIX: Colab may expose Python 3.13 without the matching ensurepip/venv
# package. The original setup.sh picks 3.13 first, then fails while bootstrapping
# pip. Find an interpreter that can actually create a venv, preferring 3.12.
import tempfile
import re

candidate_names = ["python3.12", "python3.13", "python3.11", "python3"]
working_candidate = None
tried_versions = set()
apt_update_done = False

def _venv_smoke_test(py_bin):
    temp_root = Path(tempfile.mkdtemp(prefix="srts-venv-check-", dir="/content"))
    test_venv = temp_root / "venv"
    try:
        proc = subprocess.run(
            [py_bin, "-m", "venv", str(test_venv)],
            text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=120
        )
        ok = (proc.returncode == 0 and (test_venv / "bin" / "python").exists()
              and (test_venv / "bin" / "pip").exists())
        return ok, (proc.stdout or "")[-2500:]
    except Exception as exc:
        return False, repr(exc)
    finally:
        import shutil as _shutil
        _shutil.rmtree(temp_root, ignore_errors=True)

for candidate in candidate_names:
    candidate_path = shutil.which(candidate)
    if not candidate_path:
        continue
    try:
        version = subprocess.run(
            [candidate_path, "-c", "import sys; print(f'{sys.version_info[0]}.{sys.version_info[1]}')"],
            text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=15, check=True
        ).stdout.strip()
    except Exception:
        continue
    if version in tried_versions:
        continue
    tried_versions.add(version)
    print(f"\nProvjeravam venv: {candidate} ({version}) ...")
    ok, detail = _venv_smoke_test(candidate_path)
    if not ok:
        print("Venv/ensurepip nije prošao početni test.")
        # Debian/Ubuntu packages the ensurepip bits separately. Try the exact
        # versioned package, then verify again; do not assume python3-venv is enough.
        pkg = f"python{version}-venv"
        if not apt_update_done:
            apt_update = subprocess.run(
                ["apt-get", "update", "-qq"], text=True,
                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=300
            )
            apt_update_done = True
            if apt_update.returncode != 0:
                print("apt-get update upozorenje:", (apt_update.stdout or "")[-1500:])
        apt_install = subprocess.run(
            ["apt-get", "install", "-y", "-qq", pkg], text=True,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=300
        )
        if apt_install.returncode == 0:
            ok, detail = _venv_smoke_test(candidate_path)
        else:
            print(f"Paket {pkg} nije instaliran:", (apt_install.stdout or "")[-1500:])
    if ok:
        working_candidate = candidate
        print(f"✅ Venv provjera prošla za {candidate} ({version}).")
        break
    else:
        print(f"⚠️ {candidate} nije pogodan za venv. Detalji: {detail}")

if not working_candidate:
    raise RuntimeError(
        "Nijedan Python interpreter nije mogao kreirati venv sa pip-om. "
        "Pogledaj poruke iznad; moguće je da nedostaje odgovarajući pythonX.Y-venv paket."
    )

# Change only the interpreter preference in this local checkout of setup.sh.
# All other original installation commands and pinned project dependencies remain intact.
setup_script = PROJECT_DIR / "setup.sh"
setup_text = setup_script.read_text()
match = re.search(r"for cand in ([^;]+); do", setup_text)
if not match:
    raise RuntimeError("Nisam prepoznao Python izbor u setup.sh; neću ga automatski prepravljati.")
original_candidates = [x for x in match.group(1).split() if x in candidate_names]
ordered_candidates = [working_candidate] + [x for x in original_candidates if x != working_candidate]
# Ensure standard fallbacks are preserved even if the script line changes slightly.
ordered_candidates += [x for x in candidate_names if x not in ordered_candidates]
patched_line = "for cand in " + " ".join(ordered_candidates) + "; do"
if match.group(0) != patched_line:
    setup_text = setup_text[:match.start()] + patched_line + setup_text[match.end():]
    setup_script.write_text(setup_text)
    print("Izmijenjen je samo redoslijed Python kandidata u lokalnoj kopiji setup.sh:", ordered_candidates)
else:
    print("setup.sh već ima odgovarajući redoslijed Python kandidata:", ordered_candidates)

for tool in ("python3.13", "python3.12", "python3.11", "node", "npm", "curl"):
    path = shutil.which(tool)
    if path:
        ver = subprocess.run([path, "--version"], text=True, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, timeout=15)
        print(f"{tool}: {path} | {(ver.stdout or '').strip()[:180]}")
    else:
        print(f"{tool}: NIJE PRONAĐEN")

install_env = os.environ.copy()
install_env.setdefault("SRTS_NODE_VERSION", "22.23.1")
SETUP_LOG.write_text("")
command = f"bash setup.sh 2>&1 | tee {str(SETUP_LOG)}"
print("\nPokrećem originalni setup.sh; puna dijagnostika se čuva u:", SETUP_LOG)
print("Ako instalacija ne uspije, ćelija će prikazati zadnjih 160 linija loga.\n")
result = subprocess.run(
    ["bash", "-o", "pipefail", "-c", command],
    cwd=PROJECT_DIR,
    env=install_env,
    text=True,
)

if result.returncode != 0:
    log_text = SETUP_LOG.read_text(errors="replace") if SETUP_LOG.exists() else ""
    tail = "\n".join(log_text.splitlines()[-160:])
    lower = log_text.lower()
    if "npm error" in lower or "npm err!" in lower:
        stage = "Frontend npm instalacija (provjeri npm error iz loga)."
    elif ("no matching distribution found" in lower or "could not find a version that satisfies" in lower
          or "error: subprocess-exited-with-error" in lower or "failed building wheel" in lower):
        stage = "Python backend zavisnosti / wheel (provjeri Python verziju i pip poruku)."
    elif "curl:" in lower or "checksum mismatch" in lower or "downloading node" in lower and "error" in lower:
        stage = "Preuzimanje lokalnog Node.js runtime-a ili provjera checksum-a."
    elif "could not create a venv" in lower or "no module named venv" in lower:
        stage = "Kreiranje Python virtualnog okruženja."
    elif "neither gpu nor cpu ray tracing" in lower or "mitsuba could not be imported" in lower:
        stage = "Inicijalizacija Mitsuba/Dr.Jit GPU ili LLVM backend-a."
    else:
        stage = "setup.sh je prekinut; pogledaj posljednje linije loga ispod."
    print("\n❌ Instalacija nije uspjela.")
    print("Vjerovatna faza:", stage)
    print("Log fajl:", SETUP_LOG)
    print("\n========== KRAJ SETUP LOGA (zadnjih 160 linija) ==========")
    print(tail if tail else "Log je prazan; setup.sh nije ostavio izlaz.")
    raise RuntimeError(
        f"setup.sh je završio sa kodom {result.returncode}. Uzrok nije zaključiv samo iz exit koda; "
        f"detaljan izlaz je sačuvan u {SETUP_LOG}. Ako se greška ponovi, pošalji dio loga iznad koji počinje prvom stvarnom ERROR porukom."
    )

# Reconcile Python dependencies with the manifests in the uploaded ZIP even when
# an existing .venv is reused. Pip leaves already-satisfied packages untouched.
PYTHON_FOR_SYNC = PROJECT_DIR / ".venv" / "bin" / "python"
if not PYTHON_FOR_SYNC.exists():
    raise FileNotFoundError(f"Nedostaje projektni Python: {PYTHON_FOR_SYNC}")
requirements_files = [PROJECT_DIR / "backend" / "requirements.txt"]
dev_requirements_file = PROJECT_DIR / "backend" / "requirements-dev.txt"
if dev_requirements_file.exists():
    requirements_files.append(dev_requirements_file)
for requirements_file in requirements_files:
    print(f"\nSinhronizujem Python pakete iz ZIP manifesta: {requirements_file.name}")
    subprocess.run(
        [str(PYTHON_FOR_SYNC), "-m", "pip", "install", "-r", str(requirements_file)],
        cwd=PROJECT_DIR, env=os.environ.copy(), check=True
    )

PYTHON_BIN = PROJECT_DIR / ".venv" / "bin" / "python"
NODE_BIN = PROJECT_DIR / ".node" / "bin" / "node"
if not PYTHON_BIN.exists():
    raise FileNotFoundError(f"Nedostaje projektni Python: {PYTHON_BIN}")
if not (PROJECT_DIR / "node_modules" / ".bin" / "vite").exists():
    raise FileNotFoundError("Nedostaju frontend zavisnosti (node_modules/.bin/vite). Provjeri setup log.")

print("\n✅ setup.sh je završio bez prijavljene greške.")
print("Backend Python:", PYTHON_BIN)
if NODE_BIN.exists():
    print("Projektni Node:", NODE_BIN)


## Opcionalno: uključivanje 5G NR PUSCH BER/BLER funkcionalnosti

Osnovna aplikacija radi bez dodatnog PHY paketa. Projekat navodi da je ova funkcionalnost opcionalna i da njene zavisnosti mogu zahtijevati oko 4 GB preuzimanja. Ako ti je potreban **Run BER Sweep**, promijeni `INSTALL_OPTIONAL_PHY` na `True` i pokreni ovu ćeliju **prije pokretanja servera**. Ako ti je potreban samo GUI, radio-mape, ray tracing, mobilnost i izvozi, ostavi `False`.


In [ ]:
# 4. Opcionalna instalacija PHY paketa (podrazumijevano isključena)
INSTALL_OPTIONAL_PHY = False  # promijeni u True samo ako želiš BER/BLER PUSCH sweep

from pathlib import Path
import subprocess

PROJECT_DIR = Path("/content/SionnaRTStudio")
if INSTALL_OPTIONAL_PHY:
    req = PROJECT_DIR / "backend" / "requirements-phy.txt"
    if not req.exists():
        raise FileNotFoundError(f"Nije pronađen {req}")
    print("Instaliram opcionalni Sionna PHY paket; potrebno je dodatno vrijeme i disk prostor...")
    subprocess.run([str(PROJECT_DIR / ".venv" / "bin" / "python"), "-m", "pip", "install", "-r", str(req)], check=True)
    print("✅ PHY paket instaliran. Pokreni/ponovo pokreni server tek nakon ove ćelije.")
else:
    print("Preskačem opcionalni PHY paket. Sve osnovne funkcionalnosti aplikacije ostaju dostupne; BER/BLER sweep može biti označen kao nedostupan.")


In [ ]:
# 5. Provjere instalacije, automatski svi project checks, backend testovi i frontend build
from pathlib import Path
import subprocess
import os
import sys
import json

PROJECT_DIR = Path("/content/SionnaRTStudio")
PYTHON_BIN = str(PROJECT_DIR / ".venv" / "bin" / "python")
if not Path(PYTHON_BIN).exists():
    raise FileNotFoundError(f"Projektni virtual environment nije pronađen: {PYTHON_BIN}")

package_json = PROJECT_DIR / "package.json"
if not package_json.exists():
    raise FileNotFoundError("ZIP ne sadrži package.json.")
package = json.loads(package_json.read_text(encoding="utf-8"))
project_scripts = package.get("scripts", {})

# Koristi projektni Node ako postoji.
test_env = os.environ.copy()
if (PROJECT_DIR / ".node" / "bin").exists():
    test_env["PATH"] = str(PROJECT_DIR / ".node" / "bin") + os.pathsep + test_env.get("PATH", "")

# pytest je development dependency i instalira se u projektni .venv, nikada u system Python.
probe = subprocess.run(
    [PYTHON_BIN, "-c", "import pytest"], cwd=PROJECT_DIR,
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
if probe.returncode != 0:
    print("Installing pytest into the project's .venv (not system Python)...")
    dev_requirements = PROJECT_DIR / "backend" / "requirements-dev.txt"
    pip_args = [PYTHON_BIN, "-m", "pip", "install"]
    pip_args += ["-r", str(dev_requirements)] if dev_requirements.exists() else ["pytest"]
    subprocess.run(pip_args, cwd=PROJECT_DIR, env=test_env, check=True)

version = subprocess.run(
    [PYTHON_BIN, "-c", "import sys; print(sys.executable)"],
    cwd=PROJECT_DIR, text=True, capture_output=True, check=True
)
print("Backend test interpreter:", version.stdout.strip())

# Discover every repository check from package.json. New phases can add a
# check:<name> script without requiring a new Colab notebook.
check_scripts = sorted(name for name in project_scripts if name.startswith("check:"))
print("Discovered project checks:", ", ".join(check_scripts) if check_scripts else "none")

test_cmds = [
    ("Backend pytest suite", [PYTHON_BIN, "-m", "pytest", "-q", "backend/tests"]),
]
test_cmds.extend((f"Project check: {name}", ["npm", "run", name]) for name in check_scripts)
test_cmds.extend([
    ("Frontend TypeScript check", ["npm", "run", "lint"]),
    ("Frontend production build", ["npm", "run", "build"]),
])

failed = []
for label, cmd in test_cmds:
    print(f"\n=== {label} ===")
    try:
        result = subprocess.run(
            cmd, cwd=PROJECT_DIR, env=test_env, text=True,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=600
        )
        print(result.stdout[-16000:] if result.stdout else "(bez izlaza)")
        if result.returncode == 0:
            print(f"✅ {label}: PASS")
        else:
            print(f"❌ {label}: FAIL (exit={result.returncode})")
            failed.append(label)
    except subprocess.TimeoutExpired:
        print(f"❌ {label}: timeout nakon 600 s.")
        failed.append(label)

if failed:
    raise RuntimeError("Provjere nijesu prošle: " + ", ".join(failed))
print("\n✅ Sve uključene provjere su prošle.")


In [ ]:
# 6. Pokretanje backend-a + frontend-a i provjera /api/health
# Server se pokreće u posebnoj procesnoj grupi da bi ostao aktivan nakon završetka ćelije.
import os
import sys
import time
import json
import socket
import signal
import subprocess
import urllib.request
import urllib.error
from pathlib import Path
from IPython.display import display, HTML

PROJECT_DIR = Path("/content/SionnaRTStudio")
LOG_PATH = Path("/content/SionnaRTStudio_colab.log")
FRONTEND_PORT = 3000

if not (PROJECT_DIR / ".venv" / "bin" / "python").exists():
    raise RuntimeError("Nema instaliranog .venv. Prvo uspješno pokreni setup ćeliju.")

def get_json(url, timeout=3):
    try:
        with urllib.request.urlopen(url, timeout=timeout) as response:
            raw = response.read().decode("utf-8", errors="replace")
            return json.loads(raw), response.status
    except Exception:
        return None, None

def get_status(url, timeout=3):
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "SionnaRTStudio-Colab-Launcher"})
        with urllib.request.urlopen(req, timeout=timeout) as response:
            return response.status
    except Exception:
        return None

def port_is_free(port, host="127.0.0.1"):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        return sock.connect_ex((host, port)) != 0

# Ako je app već aktivna u ovoj sesiji, iskoristi je umjesto pokretanja duplih procesa.
known_port = globals().get("BACKEND_PORT")
known_process = globals().get("SRTS_PROCESS")
known_health = None
if known_port:
    known_health, _ = get_json(f"http://127.0.0.1:{int(known_port)}/api/health")
existing_health, _ = get_json("http://127.0.0.1:8000/api/health")
existing_frontend = get_status(f"http://127.0.0.1:{FRONTEND_PORT}/")
existing_proxy_health, _ = get_json(f"http://127.0.0.1:{FRONTEND_PORT}/api/health")

if known_health and known_health.get("app") == "SionnaRTStudio" and existing_frontend == 200 and existing_proxy_health and existing_proxy_health.get("app") == "SionnaRTStudio":
    BACKEND_PORT = int(known_port)
    print("✅ SionnaRTStudio je već pokrenut u ovom kernelu; koristim postojeći server.")
elif existing_health and existing_health.get("app") == "SionnaRTStudio" and existing_frontend == 200 and existing_proxy_health and existing_proxy_health.get("app") == "SionnaRTStudio":
    BACKEND_PORT = 8000
    print("✅ SionnaRTStudio je već pokrenut; koristim postojeći server.")
else:
    if not port_is_free(FRONTEND_PORT):
        raise RuntimeError(
            f"Port {FRONTEND_PORT} je zauzet, ali ne izgleda kao aktivni SionnaRTStudio. "
            "U čistom Colab runtime-u obično je slobodan. Restartuj runtime i ponovo pokreni notebook."
        )
    free_backend_ports = [p for p in range(8000, 8020) if port_is_free(p)]
    if not free_backend_ports:
        raise RuntimeError("Nijedan backend port u opsegu 8000–8019 nije slobodan. Restartuj Colab runtime.")
    BACKEND_PORT = 8000 if 8000 in free_backend_ports else free_backend_ports[0]

    run_env = os.environ.copy()
    run_env["SRTS_BACKEND_PORT"] = str(BACKEND_PORT)
    # Vite runs behind Colab's dynamic proxy hostname. Allow only the Colab
    # runtime hostname suffix; do not disable Vite host validation globally.
    run_env["__VITE_ADDITIONAL_SERVER_ALLOWED_HOSTS"] = ".codatalab-user-runtimes.internal"
    # run.sh koristi lokalni Node ako je setup.sh instalirao .node.
    if (PROJECT_DIR / ".node" / "bin").exists():
        run_env["PATH"] = str(PROJECT_DIR / ".node" / "bin") + os.pathsep + run_env.get("PATH", "")

    log_handle = open(LOG_PATH, "ab", buffering=0)
    SRTS_LOG_HANDLE = log_handle
    SRTS_PROCESS = subprocess.Popen(
        ["bash", "run.sh"], cwd=PROJECT_DIR, env=run_env,
        stdout=log_handle, stderr=subprocess.STDOUT,
        stdin=subprocess.DEVNULL, start_new_session=True
    )
    print(f"Pokrenut run.sh (PID={SRTS_PROCESS.pid}); log: {LOG_PATH}")
    print("Čekam da backend i frontend postanu dostupni...")

    deadline = time.time() + 300
    health = None
    frontend_status = None
    while time.time() < deadline:
        health, _ = get_json(f"http://127.0.0.1:{BACKEND_PORT}/api/health", timeout=2)
        frontend_status = get_status(f"http://127.0.0.1:{FRONTEND_PORT}/", timeout=2)
        proxied_health, _ = get_json(f"http://127.0.0.1:{FRONTEND_PORT}/api/health", timeout=2)
        if health and health.get("app") == "SionnaRTStudio" and frontend_status == 200 and proxied_health and proxied_health.get("app") == "SionnaRTStudio":
            break
        if SRTS_PROCESS.poll() is not None:
            break
        time.sleep(2)

    health, _ = get_json(f"http://127.0.0.1:{BACKEND_PORT}/api/health", timeout=3)
    frontend_status = get_status(f"http://127.0.0.1:{FRONTEND_PORT}/", timeout=3)
    proxied_health, _ = get_json(f"http://127.0.0.1:{FRONTEND_PORT}/api/health", timeout=3)
    if not (health and health.get("app") == "SionnaRTStudio" and frontend_status == 200 and proxied_health and proxied_health.get("app") == "SionnaRTStudio"):
        print("\n❌ Server nije prošao health check. Posljednji dio loga:")
        try:
            print(LOG_PATH.read_text(errors="replace")[-18000:])
        except Exception as exc:
            print("Log nije moguće pročitati:", exc)
        raise RuntimeError(
            "SionnaRTStudio nije postao spreman u roku od 5 minuta. Log iznad obično pokazuje uzrok "
            "(npr. CUDA/LLVM, npm ili mrežna instalacija). Ispravi uzrok pa ponovi setup/start ćeliju."
        )

# Izvještaj health endpoint-a — verzije i GPU/CPU varijanta koju je backend stvarno aktivirao.
health, _ = get_json(f"http://127.0.0.1:{BACKEND_PORT}/api/health")
print("\n✅ Backend i frontend rade. /api/health:")
print(json.dumps(health, indent=2, ensure_ascii=False))

try:
    from google.colab import output
    app_url = output.eval_js(f"google.colab.kernel.proxyPort({FRONTEND_PORT})")
    print("\nOtvori SionnaRTStudio:", app_url)
    display(HTML(
        '<div style="padding:14px;border:1px solid #bbb;border-radius:10px;">'
        f'<a href="{app_url}" target="_blank" rel="noopener noreferrer" '
        'style="font-size:18px;font-weight:600;">🚀 Otvori SionnaRTStudio u novom tabu</a>'
        '<p style="margin-bottom:0">Ako je GUI otvoren, provjeri da status u aplikaciji pokazuje RT-CORE ONLINE.</p>'
        '</div>'
    ))
except Exception as exc:
    print("Colab proxy URL nije automatski prikazan:", exc)
    try:
        from google.colab import output
        output.serve_kernel_port_as_window(FRONTEND_PORT)
    except Exception:
        print("Otvori lokalni frontend preko Colab port proxy-ja za port 3000.")


In [ ]:
# 7. Prikaz posljednjih logova ako backend/GUI pokaže grešku
from pathlib import Path

LOG_PATH = Path("/content/SionnaRTStudio_colab.log")
if LOG_PATH.exists():
    log_text = LOG_PATH.read_text(errors="replace")
    print(log_text[-25000:] if log_text else "Log je prazan.")
else:
    print("Log još ne postoji. Prvo pokreni start ćeliju.")


In [ ]:
# 8. Zaustavi SionnaRTStudio procese kada završiš
# Ovo zaustavlja procesnu grupu koju je start ćelija pokrenula.
import os
import signal
import time

proc = globals().get("SRTS_PROCESS")
if proc is None:
    print("U ovom kernelu nema zapamćenog procesa SRTS_PROCESS. Ako je runtime restartovan, svi njegovi procesi su već ugašeni.")
elif proc.poll() is not None:
    print("SionnaRTStudio proces je već završen.")
else:
    try:
        os.killpg(proc.pid, signal.SIGTERM)
        time.sleep(2)
        if proc.poll() is None:
            # Signal cijeloj grupi; run.sh, uvicorn i Vite pripadaju istoj sesiji.
            os.killpg(proc.pid, signal.SIGKILL)
        print("✅ Zahtjev za zaustavljanje SionnaRTStudio je poslat.")
    except ProcessLookupError:
        print("Procesi su već zaustavljeni.")


## Napomene i česti problemi

- **RT-CORE OFFLINE:** pogledaj `SionnaRTStudio_colab.log` i rezultat `/api/health`. Backend mora uspješno inicijalizovati Mitsuba varijantu.
- **CPU umjesto GPU:** koristi `Runtime → Change runtime type → GPU`, restartuj runtime i ponovi notebook. Ako GPU nije dostupan, projekat može pokušati LLVM CPU fallback; ray tracing će biti sporiji.
- **Pogrešan source ZIP:** provjeri SHA-256 i naziv ZIP-a iz ćelije 2; notebook ne klonira niti ažurira GitHub.
- **Instalacija ne završi:** Colab mreža ili pip/npm mirror mogu privremeno biti nedostupni. Pokreni ponovo neuspjelu ćeliju; ne uploaduj drugi source ZIP dok je server aktivan; za novu verziju restartuj runtime i ponovo pokreni notebook.
- **BER/BLER dugme nije dostupno:** to je opcionalni Sionna PHY paket. Instaliraj ga zasebnom ćelijom prije pokretanja servera, pa restartuj server.
- **Sesija je istekla:** Colab nije stalni hosting; ponovo poveži runtime i pokreni notebook redom.
- **Privatnost:** ne koristi aplikaciju za osjetljive podatke. Port proxy je namijenjen tvojoj Colab sesiji, ali aplikacija sama nije dodatno zaštićena autentikacijom.

Ovaj notebook ne preuzima izvorni kod sa GitHub-a. Za svaku sesiju koristi se tačno ZIP arhiva koju uploaduješ; SHA-256 arhive ispisuje se u ćeliji 2 radi identifikacije verzije. Start ćelija ispisuje stvarno stanje `/api/health` radi dijagnostike.
